Read Bronze

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 3, Finished, Available, Finished, False)

In [2]:
calendar_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/calendar.csv")

sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sales_train_validation.csv")

prices_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/Bronze/sell_prices.csv")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 4, Finished, Available, Finished, False)

In [3]:
silver_calendar = calendar_df \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("event_name_1", coalesce(col("event_name_1"), lit("None"))) \
    .withColumn("event_type_1", coalesce(col("event_type_1"), lit("None"))) \
    .withColumn("event_name_2", coalesce(col("event_name_2"), lit("None"))) \
    .withColumn("event_type_2", coalesce(col("event_type_2"), lit("None"))) \
    .withColumn("snap_CA", col("snap_CA").cast("integer")) \
    .withColumn("snap_TX", col("snap_TX").cast("integer")) \
    .withColumn("snap_WI", col("snap_WI").cast("integer"))

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 5, Finished, Available, Finished, False)

In [4]:
silver_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_calendar")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 6, Finished, Available, Finished, False)

In [5]:
silver_prices = prices_df \
    .withColumn("wm_yr_wk", col("wm_yr_wk").cast("integer")) \
    .withColumn("sell_price", col("sell_price").cast("double"))

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 7, Finished, Available, Finished, False)

In [6]:
silver_prices.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_prices")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 8, Finished, Available, Finished, False)

In [7]:
day_columns = [
    c for c in sales_df.columns
    if c.startswith("d_")
]

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 9, Finished, Available, Finished, False)

In [8]:
stack_expression = ", ".join(
    [f"'{c}', `{c}`" for c in day_columns]
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 10, Finished, Available, Finished, False)

In [9]:
base_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 11, Finished, Available, Finished, False)

In [10]:
silver_sales = sales_df.select(
    *base_columns,
    expr(
        f"stack({len(day_columns)}, {stack_expression}) as (d, sales)"
    )
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 12, Finished, Available, Finished, False)

In [11]:
silver_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_sales")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 13, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 9a29e127-5a20-4a83-a4ea-73737f51a835).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- date: date (nullable = true)
-- id: string (nullable = true)
-- item_id: string (nullable = true)
-- dept_id: string (nullable = true)
-- cat_id: string (nullable = true)
-- store_id: string (nullable = true)
-- state_id: string (nullable = true)
-- sales: integer (nullable = true)


Data schema:
root
-- id: string (nullable = true)
-- item_id: string (nullable = true)
-- dept_id: string (nullable = true)
-- cat_id: string (nullable = true)
-- store_id: string (nullable = true)
-- state_id: string (nullable = true)
-- d: st

In [12]:
silver_calendar = spark.table("silver_calendar")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 14, Finished, Available, Finished, False)

In [13]:
window_spec = Window.orderBy("date")

gold_calendar = silver_calendar \
    .withColumn(
        "day_number",
        row_number().over(window_spec)
    ) \
    .withColumn(
        "d",
        concat(lit("d_"), col("day_number"))
    )

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 15, Finished, Available, Finished, False)

In [14]:
dim_date = gold_calendar.select(
    "d",
    "date",
    "wm_yr_wk",
    "year",
    "month",
    "weekday",
    "event_name_1",
    "event_type_1"
).dropDuplicates()

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 16, Finished, Available, Finished, False)

In [15]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_date")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 17, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 9d3c1e7b-eee2-44ec-b241-103c39a49976).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- date: date (nullable = true)
-- wm_yr_wk: integer (nullable = true)
-- weekday: string (nullable = true)
-- wday: integer (nullable = true)
-- month: integer (nullable = true)
-- year: integer (nullable = true)
-- event_name_1: string (nullable = true)
-- event_type_1: string (nullable = true)
-- event_name_2: string (nullable = true)
-- event_type_2: string (nullable = true)
-- snap_CA: integer (nullable = true)
-- snap_TX: integer (nullable = true)
-- snap_WI: integer (nullable = true)
-- day_of_month: integer (nullable =

In [16]:
silver_sales = spark.table("silver_sales")

dim_product = silver_sales.select(
    "item_id",
    "dept_id",
    "cat_id"
).dropDuplicates()

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 18, Finished, Available, Finished, False)

In [17]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_product")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 19, Finished, Available, Finished, False)

In [18]:
dim_store = silver_sales.select(
    "store_id",
    "state_id"
).dropDuplicates()

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 20, Finished, Available, Finished, False)

In [19]:
dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_dim_store")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 21, Finished, Available, Finished, False)

In [20]:
silver_prices = spark.table("silver_prices")

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 22, Finished, Available, Finished, False)

In [24]:
fact_sales = silver_sales.join(
    gold_calendar.select(
        "date",
        "d",
        "wm_yr_wk",
        "year",
        "month",
        "weekday",
        "event_name_1",
        "event_type_1"
    ),
    on="date",
    how="left"
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 26, Finished, Available, Finished, False)

In [25]:
display(
    fact_sales.select(
        "date",
        "d",
        "item_id",
        "store_id",
        "sales",
        "wm_yr_wk"
    ).limit(20)
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1983eaf5-830d-43f1-86de-9c7c0bd8115a)

In [26]:
fact_sales = fact_sales.join(
    silver_prices.select(
        "store_id",
        "item_id",
        "wm_yr_wk",
        "sell_price"
    ),
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 28, Finished, Available, Finished, False)

In [27]:
display(
    fact_sales.select(
        "date",
        "item_id",
        "store_id",
        "sales",
        "sell_price"
    ).limit(20)
)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a10d9095-19a0-4893-916e-9407378a5d82)

In [31]:
test_record = fact_sales.select(
    "item_id",
    "store_id",
    "wm_yr_wk"
).filter(
    col("sell_price").isNull()
).limit(1)

display(test_record)

StatementMeta(, 1c15faf2-7e81-414f-9405-9d8c94f75c24, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2801962b-555d-432d-9153-6bb963fc4d99)